In [ ]:
import csv
import requests
from io import StringIO
from google.colab import files

SHEET_ID = "118Znzcf8Nr2y0-YPCnO3JbZX-QZhm39q7ATDTpouVjE"
GID = ""  # gid of the Attachments tab (the number after #gid= in the tab's URL)

SHEET_URL = (
    f"https://docs.google.com/spreadsheets/d/"
    f"{SHEET_ID}/export?format=csv&gid={GID}"
)

response = requests.get(SHEET_URL)
response.raise_for_status()

reader = list(csv.DictReader(StringIO(response.content.decode("utf-8")), delimiter=","))


In [ ]:
ID = "id"

# Only balance lives in the sheet; names, icons, components and pages stay in FEATURE_AttachmentItems.lua.
SIZES = {"small", "medium", "large"}
STRING_KEYS = {"size"}
NUMBER_KEYS = {"cost", "tier", "weight", "max_stock", "scrap"}
REQUIRED_KEYS = {"size", "cost", "tier"}

# Every header column except those starting with "#"; prefix a column with "#" to drop it.
header = reader[0].keys() if reader else []
KEYS = [k for k in header if k != ID and not k.startswith("#")]
unknown = [k for k in KEYS if k not in STRING_KEYS | NUMBER_KEYS]
if unknown:
    raise ValueError(f"unknown columns (prefix them with #): {unknown}")


def format_value(key, raw_value, item_id):
    """Cell value -> Lua expression; None omits the field."""
    value = raw_value.strip()
    if value == "":
        if key in REQUIRED_KEYS:
            raise ValueError(f"{item_id}.{key} is required")
        return None
    if key == "size":
        if value not in SIZES:
            raise ValueError(f'{item_id}.size: expected small/medium/large, got "{value}"')
        return f'"{value}"'
    number = value.replace(",", "")
    if not number.lstrip("-").isdigit():
        raise ValueError(f'{item_id}.{key}: expected an integer, got "{value}"')
    return number


def get_item_lua_snippet(row):
    fields = []
    for key in KEYS:
        lua_value = format_value(key, row.get(key) or "", row[ID])
        if lua_value is not None:
            fields.append(f"{key} = {lua_value}")
    return f"    {row[ID]} = {{" + ", ".join(fields) + "}"


In [ ]:
LUA_FILE_NAME = "FEATURE_AttachmentBalance.lua"

rows = []
seen = set()
for row in reader:
    item_id = (row.get(ID) or "").strip()
    if not item_id:
        continue
    if item_id in seen:
        raise ValueError(f'duplicate id: "{item_id}"')
    seen.add(item_id)
    row[ID] = item_id
    rows.append(row)

snippets = [get_item_lua_snippet(row) for row in rows]
print(f"{len(snippets)} attachments generated.")

with open(LUA_FILE_NAME, "w", encoding="utf-8", newline="\n") as lua_file:
    lua_file.write("-- ========== THIS IS AN AUTOMATICALLY GENERATED FILE! ==========\n")
    lua_file.write("-- Source: attachments sheet via GBO_AttachmentsToLua.ipynb. Edit the sheet, not this file.\n\n")
    lua_file.write("RAT_ATT_BALANCE = {\n")
    lua_file.write(",\n".join(snippets))
    lua_file.write("\n}\n")

print("\n".join(snippets[:3]))
files.download(LUA_FILE_NAME)
